# Text-to-SQL fine-tuning notebook

This notebook is the GPU-first entry point for the project. It installs dependencies, pulls the repo, checks the runtime, evaluates the baseline, fine-tunes the model, and reports the before/after metrics.

In [ ]:
# Step 1: install the required packages.
# The current official install is simply `pip install unsloth`; older forms are stale.
!pip install --quiet unsloth
!pip install --quiet datasets transformers trl peft bitsandbytes accelerate

# Step 2: clone or update the project repository.
# Replace the placeholder below with your public GitHub repo URL before running.
REPO_URL = "https://github.com/your-username/your-repo.git"
!git clone "$REPO_URL" repo || true
!cd repo && git pull || true

import os, sys
repo_path = os.path.abspath('/content/repo') if os.path.exists('/content/repo') else os.path.abspath('repo')
if os.path.exists(repo_path):
    sys.path.insert(0, repo_path)

print('Repo ready for import from:', repo_path)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Switch the notebook runtime to a GPU-backed option in Colab or Kaggle.')

In [ ]:
from datasets import load_dataset
from data import load_sql_dataset, render_prompt

train_ds, test_ds = load_sql_dataset(train_subset=200)
print('Train size:', len(train_ds))
print('Test size:', len(test_ds))
row = train_ds[0]
print('Example row keys:', list(row.keys()))
print('Example question:', row['question'])
print('Example schema:', row['context'][:200])

In [ ]:
from train import load_base_model
from evaluate import run_evaluation

model, tokenizer = load_base_model()
base_metrics = run_evaluation(model, tokenizer, test_ds, n=20)
print('BASELINE metrics:', base_metrics)

In [ ]:
from train import attach_lora, train_model

model = attach_lora(model)
trainer = train_model(model, tokenizer, train_ds)
print('LoRA training finished.')

In [ ]:
from evaluate import run_evaluation

fine_tuned_metrics = run_evaluation(model, tokenizer, test_ds, n=20)
print('FINE-TUNED metrics:', fine_tuned_metrics)

print('\nBefore vs After comparison')
print({'baseline': base_metrics, 'fine_tuned': fine_tuned_metrics})

This final section saves the adapter and shows a single inference example using a hand-written schema and question.

In [ ]:
from config import ADAPTER_DIR

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved adapter to:', ADAPTER_DIR)

question = "List all users with names starting with A."
schema = "CREATE TABLE users (id INTEGER, name TEXT);"
prompt = render_prompt(question, schema)
print('\nInference prompt preview:')
print(prompt)